# 03 — Build the dataset · my pipeline


This thesis's preprocessing. Every deviation from the authors is argued in
`pipelines/thesis/preprocessing.py`, with the measurement that motivated it — and,
where the measurement went against the proposal, that is stated too.

| step | rule | why |
|---|---|---|
| slices | 8 spread, trimming 15% each end | neighbouring slices are near-duplicates |
| crop | **80 mm physical**, constant 0.357 mm/px | a proportional crop erases tumour size |
| normalisation | min-max over the whole volume | preserves enhancement kinetics |
| cohorts | I-SPY2 only | the source probe reaches 0.9978 on pooled data |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dataset_config as config
from dataset_config import Config, TASKS

plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False})

from core.dataset_builder import build, patient_table

## Configure

Everything comes from `config.py`. Change it there, not here.

In [ ]:
# The thesis target: three-class molecular subtype, I-SPY2 only.
CONFIGS = [Config(pipeline="mine", task="subtype", cohorts=("spy2",))]

BUILD_ARGS = dict(n_slices=8, trim_fraction=0.15, crop_mm=80.0,
                  save_size=224, normalization="minmax", workers=8)

for c in CONFIGS:
    print(f"{c.task:<8} {c.num_classes} classes  {', '.join(c.class_names)}")

## Dry run first

Prints the plan and the patient counts without writing a single file. Always check this before spending disk.

In [ ]:
for cfg in CONFIGS:
    print("=" * 78)
    build(cfg, dry_run=True, **BUILD_ARGS)
    print()

## Build

Each build ends with the leakage verification and refuses to pass if a patient appears in two splits.

In [ ]:
for cfg in CONFIGS:
    if (cfg.dataset_dir / "metadata.csv").is_file():
        print(f"{cfg.dataset_dir.name}: already built — delete the folder to rebuild")
        continue
    print("=" * 78)
    build(cfg, **BUILD_ARGS)
    print()

## What came out

In [ ]:
for cfg in CONFIGS:
    f = cfg.dataset_dir / "metadata.csv"
    if not f.is_file():
        continue
    m = pd.read_csv(f); p = m.drop_duplicates("pid")
    print(f"### {cfg.dataset_dir.name}")
    print(f"  {len(m):,} images · {p.pid.nunique():,} patients · "
          f"{len(m)/p.pid.nunique():.2f} slices per patient")
    print(pd.crosstab(p.split, p.label_name, margins=True).to_string())
    base = p[p.split == 'test'].label_name.value_counts()
    print(f"  trivial baseline accuracy on test: {base.max()}/{base.sum()} = "
          f"{base.max()/base.sum():.4f}\n")

## Sample images

In [ ]:
from PIL import Image
cfg = CONFIGS[0]
m = pd.read_csv(cfg.dataset_dir / "metadata.csv")
sample = m.groupby("label_name").head(4)
fig, axes = plt.subplots(len(sample.label_name.unique()), 4,
                         figsize=(10, 2.6 * sample.label_name.nunique()))
axes = np.atleast_2d(axes)
for r, (name, g) in enumerate(sample.groupby("label_name")):
    for c, (_, row) in enumerate(g.iterrows()):
        ax = axes[r, c]
        ax.imshow(Image.open(cfg.dataset_dir / "images" / row.filename))
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        if c == 0:
            ax.set_ylabel(name, fontsize=9)
        ax.set_title(f"{row.pid}  z={row.slice_index}", fontsize=7)
plt.tight_layout(); plt.show()